In [1]:
# -*- coding: utf-8 -*-
"""
Two-stage XGBoost + Optuna + Prediction + Monte Carlo CI
改为预测 2015 / 2017 / 2019 / 2021 / 2023

说明:
- Stage 1: static features -> predict y_log1p
- Stage 2: dynamic features + stage1_pred_log1p -> predict residual_log1p
- Train in log1p space
- Evaluation:
  1) Train OOF by city (GroupKFold)
  2) Holdout new city (GroupShuffleSplit)
- Optional calibration in log space
- Exports:
  metrics_summary_stagewise.csv
  train_oof_predictions_by_city.csv
  holdout_predictions_newcity.csv
  all_predictions.csv
  stage1_model.json / stage2_model.json / train_artifacts.json
  predictions_2015_monthly.csv
  predictions_2017_monthly.csv
  predictions_2019_monthly.csv
  predictions_2021_monthly.csv
  predictions_2023_monthly.csv
  predictions_2015_monthly_with_ci_mc.csv
  predictions_2017_monthly_with_ci_mc.csv
  predictions_2019_monthly_with_ci_mc.csv
  predictions_2021_monthly_with_ci_mc.csv
  predictions_2023_monthly_with_ci_mc.csv

约定:
- 训练CSV: A列=city, B列=y(浓度), C~Z为特征
- 预测CSV: 至少需要包含训练时那批特征列名(按列名匹配), A列city可有可无
"""

import os
import re
import json
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import xgboost as xgb
import optuna

warnings.filterwarnings("ignore")


# =========================
# 0) 只需要改这里
# =========================
DATA_PATH = r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\20260205气相-Cl预测无能源消耗无轻重工业无经纬度.csv"

PRED_2015 = r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2015年预测.csv"
PRED_2017 = r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2017年预测.csv"
PRED_2019 = r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2019年预测.csv"
PRED_2021 = r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2021年预测.csv"
PRED_2023 = r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\城市大气中XPAHs的存在特征\2023年预测.csv"

OUT_DIR = r"E:\HuaweiMoveData\Users\HAUWEI\Desktop\xgb_runs\two_stage_newcity_paperready_v4_2015_2017"

RANDOM_STATE = 42

TEST_SIZE_CITIES = 0.20
OOF_FOLDS = 5

N_TRIALS = 300
TIMEOUT_SEC = None

EARLY_STOPPING = 80
MAX_BOOST_ROUNDS_CAP = 4000

CLIP_NONNEG = True

MIX_W_LOG = 0.70
MIX_W_RAW = 0.30

USE_CALIBRATION = True
CALIBRATION_SPACE = "log"  # "log" or "raw"

WEIGHT_MODE = "quantile"   # "none" | "pow" | "quantile"
W_ALPHA_RANGE = (0.2, 2.0)
W_K_RANGE = (0.5, 2.5)
WEIGHT_CAP_RANGE = (5.0, 30.0)

ADD_LOG_FEATURES = True
LOG_FEATURE_KEYWORDS = [
    "gdp", "industrial", "enterprise", "enterprises", "emission", "emissions",
    "population", "added_value", "value", "ownership", "automobile"
]
ADD_CYCLIC_QUARTER = True

DROP_CONSTANT = True
NEAR_ZERO_VAR_THRESH = 1e-12
DROP_HIGH_CORR = True
HIGH_CORR_THRESH = 0.98

RUN_PREDICTION_AFTER_TRAINING = True

RUN_MC_AFTER_PRED = True
MC_N_BINS = 10
MC_N = 2000
MC_SEED = 42


# =========================
# 1) 工具函数
# =========================
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def safe_path(path: str) -> str:
    if not os.path.exists(path):
        return path
    base, ext = os.path.splitext(path)
    k = 1
    while True:
        cand = f"{base}__{k}{ext}"
        if not os.path.exists(cand):
            return cand
        k += 1


def safe_to_csv(df: pd.DataFrame, path: str, **kwargs):
    try:
        df.to_csv(path, **kwargs)
        return path
    except PermissionError:
        path2 = safe_path(path)
        df.to_csv(path2, **kwargs)
        return path2


def try_read_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到文件: {path}")

    encodings = ["utf-8-sig", "utf-8", "gb18030", "gbk", "cp1252", "latin1"]
    seps = [None, ",", "\t", ";", "|"]

    last_err = None
    for enc in encodings:
        for sep in seps:
            try:
                if sep is None:
                    return pd.read_csv(path, encoding=enc, sep=None, engine="python")
                return pd.read_csv(path, encoding=enc, sep=sep, engine="python")
            except Exception as e:
                last_err = e
                continue

    try:
        return pd.read_csv(path, encoding="latin1", sep=None, engine="python")
    except Exception as e:
        last_err = e

    raise RuntimeError(f"读取CSV失败: {path}\n最后一次错误: {repr(last_err)}")


def sanitize_cols(cols: List[str]) -> List[str]:
    out = []
    for c in cols:
        c2 = str(c).strip().replace("\ufeff", "")
        c2 = re.sub(r"[^\w\.\-\(\)]+", "_", c2)
        c2 = re.sub(r"_+", "_", c2).strip("_")
        out.append(c2)
    return out


def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    for c in df2.columns:
        if df2[c].dtype == "bool":
            df2[c] = df2[c].astype(int)
            continue
        if df2[c].dtype == "object":
            s = df2[c].astype(str).str.strip()
            s_low = s.str.lower()
            if s_low.isin(["true", "false", "yes", "no", "y", "n"]).mean() > 0.9:
                df2[c] = s_low.map({
                    "true": 1, "yes": 1, "y": 1,
                    "false": 0, "no": 0, "n": 0
                }).astype(float)
            else:
                df2[c] = pd.to_numeric(df2[c], errors="coerce")
    return df2


def safe_rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def build_dmatrix(X: np.ndarray, y: Optional[np.ndarray] = None, w: Optional[np.ndarray] = None) -> xgb.DMatrix:
    if y is None:
        return xgb.DMatrix(X)
    if w is None:
        return xgb.DMatrix(X, label=y)
    return xgb.DMatrix(X, label=y, weight=w)


def drop_constant_and_near_zero_var(Xdf: pd.DataFrame, thresh: float = 1e-12) -> Tuple[pd.DataFrame, List[str]]:
    nunique = Xdf.nunique(dropna=False)
    keep = nunique[nunique > 1].index.tolist()
    dropped_1 = [c for c in Xdf.columns if c not in keep]
    Xdf2 = Xdf[keep].copy()

    v = Xdf2.var(axis=0, ddof=0)
    keep2 = v[(v > thresh) | (~np.isfinite(v))].index.tolist()
    dropped_2 = [c for c in Xdf2.columns if c not in keep2]
    return Xdf2[keep2].copy(), (dropped_1 + dropped_2)


def drop_highly_correlated(Xdf: pd.DataFrame, thresh: float = 0.98) -> Tuple[pd.DataFrame, List[str]]:
    if Xdf.shape[1] < 2:
        return Xdf.copy(), []
    corr = Xdf.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > thresh).any()]
    return Xdf.drop(columns=to_drop), to_drop


def add_light_feature_engineering(Xdf: pd.DataFrame) -> pd.DataFrame:
    X = Xdf.copy()

    if ADD_LOG_FEATURES:
        for c in list(X.columns):
            lc = c.lower()
            if any(k in lc for k in LOG_FEATURE_KEYWORDS):
                v = X[c].astype(float).values
                X[f"log1p_{c}"] = np.log1p(np.clip(v, 0.0, None))

    if ADD_CYCLIC_QUARTER and ("quarter" in X.columns):
        q = X["quarter"].astype(float).values
        q_mod = np.where(q >= 1, q - 1, q)
        ang = 2.0 * np.pi * (q_mod / 4.0)
        X["quarter_sin"] = np.sin(ang)
        X["quarter_cos"] = np.cos(ang)

    return X


# =========================
# 2) 列映射
# =========================
def map_columns_A_B_C_to_Z(df: pd.DataFrame):
    cols = df.columns.tolist()
    if len(cols) < 3:
        raise ValueError("文件列数不足: 至少需要A(city), B(y), C(特征起)")
    id_col = cols[0]
    y_col = cols[1]
    start_idx = 2
    end_idx = min(25, len(cols) - 1)  # C~Z
    feat_cols = cols[start_idx:end_idx + 1]
    if len(feat_cols) < 1:
        raise ValueError("未找到C~Z特征列, 请检查文件")
    return id_col, y_col, feat_cols


# =========================
# 3) static / dynamic 分组
# =========================
STATIC_KEYWORDS = [
    "gdp", "industrial", "enterprise", "enterprises", "added_value", "value",
    "population", "per_capital", "per_capita", "ownership", "automobile",
    "coastal", "heating", "heavy_industrial", "area", "altitude", "proportion"
]

DYNAMIC_KEYWORDS = [
    "pm2", "pm10", "no2", "so2", "co", "o3",
    "temperature", "humidity", "wind", "precipitation", "blh", "ventilation",
    "quarter", "sin", "cos"
]


def split_static_dynamic(feature_names: List[str]) -> Tuple[List[str], List[str]]:
    static, dynamic = [], []
    for c in feature_names:
        lc = c.lower()
        is_static = any(k in lc for k in STATIC_KEYWORDS)
        is_dynamic = any(k in lc for k in DYNAMIC_KEYWORDS)

        if is_dynamic and not is_static:
            dynamic.append(c)
        elif is_static and not is_dynamic:
            static.append(c)
        elif is_static and is_dynamic:
            dynamic.append(c)
        else:
            static.append(c)

    def uniq(xs):
        seen = set()
        out = []
        for x in xs:
            if x not in seen:
                out.append(x)
                seen.add(x)
        return out

    return uniq(static), uniq(dynamic)


# =========================
# 4) 权重
# =========================
def make_weights_pow(y_raw: np.ndarray, alpha: float, cap: float) -> np.ndarray:
    w = (y_raw + 1.0) ** float(alpha)
    w = w / np.mean(w)
    w = np.clip(w, 1.0 / float(cap), float(cap))
    return w.astype(float)


def make_weights_quantile(y_raw: np.ndarray, k: float, cap: float) -> np.ndarray:
    r = pd.Series(y_raw).rank(method="average").values
    q = (r - 0.5) / len(r)
    w = np.exp(float(k) * (q - 0.5))
    w = w / np.mean(w)
    w = np.clip(w, 1.0 / float(cap), float(cap))
    return w.astype(float)


def make_weights(y_raw: np.ndarray, mode: str, w_alpha: float, w_k: float, cap: float) -> Optional[np.ndarray]:
    if mode == "none":
        return None
    if mode == "pow":
        return make_weights_pow(y_raw, w_alpha, cap)
    if mode == "quantile":
        return make_weights_quantile(y_raw, w_k, cap)
    return None


# =========================
# 5) 校准
# =========================
def calibrate_predictions(y_true_raw, pred_log, pred_raw, space="log"):
    y_true_raw = np.asarray(y_true_raw).reshape(-1)
    pred_log = np.asarray(pred_log).reshape(-1)
    pred_raw = np.asarray(pred_raw).reshape(-1)

    if space == "log":
        x = pred_log.reshape(-1, 1)
        y = np.log1p(y_true_raw).reshape(-1, 1)
    else:
        x = pred_raw.reshape(-1, 1)
        y = y_true_raw.reshape(-1, 1)

    X = np.c_[np.ones(len(x)), x]
    beta = np.linalg.lstsq(X, y, rcond=None)[0].reshape(-1)
    return beta


def apply_calibration(pred_log, pred_raw, beta, space="log"):
    pred_log = np.asarray(pred_log).reshape(-1)
    pred_raw = np.asarray(pred_raw).reshape(-1)

    if space == "log":
        pred_log_cal = beta[0] + beta[1] * pred_log
        pred_raw_cal = np.expm1(pred_log_cal)
    else:
        pred_raw_cal = beta[0] + beta[1] * pred_raw
        if CLIP_NONNEG:
            pred_raw_cal = np.clip(pred_raw_cal, 0.0, None)
        pred_log_cal = np.log1p(np.clip(pred_raw_cal, 0.0, None))

    return pred_log_cal, pred_raw_cal


# =========================
# 6) XGB训练与Optuna
# =========================
@dataclass
class CVResult:
    raw_r2: float
    log_r2: float
    best_iter: int
    score: float


def train_xgb_log(
    params: Dict,
    Xtr: np.ndarray,
    ytr_log: np.ndarray,
    Xva: np.ndarray,
    yva_log: np.ndarray,
    wtr: Optional[np.ndarray],
    num_round: int,
    early_stopping: int
) -> Tuple[xgb.Booster, int]:
    dtr = build_dmatrix(Xtr, ytr_log, w=wtr)
    dva = build_dmatrix(Xva, yva_log)

    bst = xgb.train(
        params=params,
        dtrain=dtr,
        num_boost_round=num_round,
        evals=[(dtr, "train"), (dva, "valid")],
        verbose_eval=False,
        early_stopping_rounds=early_stopping,
    )
    best_it = int(bst.best_iteration + 1)
    return bst, best_it


def eval_one_split_two_stage(
    params1: Dict,
    params2: Dict,
    X_static_tr: np.ndarray,
    X_static_va: np.ndarray,
    X_dyn_tr: np.ndarray,
    X_dyn_va: np.ndarray,
    y_tr_raw: np.ndarray,
    y_va_raw: np.ndarray,
    w_mode: str,
    w_alpha: float,
    w_k: float,
    w_cap: float,
    fixed_rounds: int,
    early_stopping: int
) -> CVResult:
    ytr_log = np.log1p(y_tr_raw)
    yva_log = np.log1p(y_va_raw)

    wtr = make_weights(y_tr_raw, w_mode, w_alpha, w_k, w_cap)
    bst1, it1 = train_xgb_log(
        params1, X_static_tr, ytr_log, X_static_va, yva_log, wtr,
        num_round=min(MAX_BOOST_ROUNDS_CAP, fixed_rounds),
        early_stopping=early_stopping
    )

    pred1_va_log = bst1.predict(build_dmatrix(X_static_va), iteration_range=(0, it1))
    pred1_tr_log = bst1.predict(build_dmatrix(X_static_tr), iteration_range=(0, it1))

    resid_tr = ytr_log - pred1_tr_log
    resid_va = yva_log - pred1_va_log

    X2_tr = np.c_[X_dyn_tr, pred1_tr_log.reshape(-1, 1)]
    X2_va = np.c_[X_dyn_va, pred1_va_log.reshape(-1, 1)]

    wtr2 = make_weights(y_tr_raw, w_mode, w_alpha, w_k, w_cap)
    bst2, it2 = train_xgb_log(
        params2, X2_tr, resid_tr, X2_va, resid_va, wtr2,
        num_round=min(MAX_BOOST_ROUNDS_CAP, fixed_rounds),
        early_stopping=early_stopping
    )

    pred2_va_resid = bst2.predict(build_dmatrix(X2_va), iteration_range=(0, it2))
    pred_va_log = pred1_va_log + pred2_va_resid
    pred_va_raw = np.expm1(pred_va_log)
    if CLIP_NONNEG:
        pred_va_raw = np.clip(pred_va_raw, 0.0, None)

    raw_r2 = float(r2_score(y_va_raw, pred_va_raw))
    log_r2 = float(r2_score(yva_log, pred_va_log))
    score = float(MIX_W_LOG * log_r2 + MIX_W_RAW * raw_r2)

    return CVResult(
        raw_r2=raw_r2,
        log_r2=log_r2,
        best_iter=int(np.median([it1, it2])),
        score=score
    )


def make_xgb_params(trial: optuna.Trial, tag: str) -> Dict:
    grow_policy = trial.suggest_categorical(f"{tag}_grow_policy", ["depthwise", "lossguide"])

    if grow_policy == "depthwise":
        max_depth = trial.suggest_int(f"{tag}_max_depth", 2, 10)
        max_leaves = 0
    else:
        max_depth = trial.suggest_int(f"{tag}_max_depth", 0, 5)
        max_leaves = trial.suggest_int(f"{tag}_max_leaves", 32, 768)

    use_alpha = trial.suggest_categorical(f"{tag}_use_alpha", [0, 1])
    alpha = trial.suggest_float(f"{tag}_alpha", 1e-8, 0.8, log=True) if use_alpha else 0.0

    params = dict(
        booster="gbtree",
        objective="reg:squarederror",
        tree_method="hist",
        eval_metric="rmse",
        seed=RANDOM_STATE,
        nthread=max(1, (os.cpu_count() or 4) - 1),

        grow_policy=grow_policy,
        max_depth=int(max_depth),
        max_leaves=int(max_leaves),

        eta=trial.suggest_float(f"{tag}_eta", 0.004, 0.10, log=True),
        subsample=trial.suggest_float(f"{tag}_subsample", 0.60, 0.95),
        colsample_bytree=trial.suggest_float(f"{tag}_colsample_bytree", 0.55, 0.95),
        colsample_bylevel=trial.suggest_float(f"{tag}_colsample_bylevel", 0.55, 0.95),
        colsample_bynode=trial.suggest_float(f"{tag}_colsample_bynode", 0.55, 0.95),

        min_child_weight=trial.suggest_float(f"{tag}_min_child_weight", 0.8, 30.0, log=True),
        gamma=trial.suggest_float(f"{tag}_gamma", 0.0, 8.0),
        max_delta_step=trial.suggest_float(f"{tag}_max_delta_step", 0.0, 12.0),

        alpha=float(alpha),
        lambda_=trial.suggest_float(f"{tag}_lambda_", 0.2, 120.0, log=True),
        max_bin=trial.suggest_int(f"{tag}_max_bin", 128, 512),
    )
    return params


def finalize_params(p: Dict) -> Dict:
    p2 = dict(
        booster="gbtree",
        objective="reg:squarederror",
        tree_method="hist",
        eval_metric="rmse",
        seed=RANDOM_STATE,
        nthread=max(1, (os.cpu_count() or 4) - 1),
    )
    for kk, vv in p.items():
        p2[kk] = vv
    if "alpha" not in p2:
        p2["alpha"] = 0.0
    if "lambda" in p2 and "lambda_" not in p2:
        p2["lambda_"] = p2.pop("lambda")
    return p2


# =========================
# 7) 预处理工件
# =========================
@dataclass
class PrepArtifacts:
    id_col: str
    y_col: str
    feature_cols_base: List[str]
    allnan_cols_train: List[str]
    medians_train_base: Dict[str, float]
    dropped_const_nzv_cols: List[str]
    dropped_corr_cols: List[str]
    final_feature_names: List[str]
    static_cols: List[str]
    dynamic_cols: List[str]


def preprocess_training(df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, pd.DataFrame, PrepArtifacts]:
    df = df.copy()
    df.columns = sanitize_cols(df.columns)
    df = df.dropna(axis=0, how="all").reset_index(drop=True)

    id_col, y_col, feat_cols = map_columns_A_B_C_to_Z(df)

    groups = df[id_col].astype(str).values
    y = pd.to_numeric(df[y_col], errors="coerce").values
    ok_y = np.isfinite(y)

    Xdf_base = df.loc[:, feat_cols].copy()
    Xdf_base = to_numeric_df(Xdf_base)

    allnan_cols_train = [c for c in Xdf_base.columns if Xdf_base[c].isna().all()]
    if allnan_cols_train:
        Xdf_base = Xdf_base.drop(columns=allnan_cols_train)

    medians_train_base = {}
    for c in Xdf_base.columns:
        v = Xdf_base[c].values.astype(float)
        med = np.nanmedian(v)
        if not np.isfinite(med):
            med = 0.0
        medians_train_base[c] = float(med)
        if np.isnan(v).any():
            Xdf_base[c] = Xdf_base[c].fillna(med)

    Xdf_base = Xdf_base.loc[ok_y].reset_index(drop=True)
    groups = groups[ok_y]
    y = y[ok_y]
    df_ok = df.loc[ok_y].reset_index(drop=True)

    Xdf = add_light_feature_engineering(Xdf_base)

    dropped_const_nzv_cols = []
    if DROP_CONSTANT:
        Xdf, dropped_const_nzv_cols = drop_constant_and_near_zero_var(Xdf, thresh=NEAR_ZERO_VAR_THRESH)

    dropped_corr_cols = []
    if DROP_HIGH_CORR:
        Xdf, dropped_corr_cols = drop_highly_correlated(Xdf, thresh=HIGH_CORR_THRESH)

    final_feature_names = Xdf.columns.tolist()
    static_cols, dynamic_cols = split_static_dynamic(final_feature_names)

    artifacts = PrepArtifacts(
        id_col=id_col,
        y_col=y_col,
        feature_cols_base=list(Xdf_base.columns),
        allnan_cols_train=allnan_cols_train,
        medians_train_base=medians_train_base,
        dropped_const_nzv_cols=dropped_const_nzv_cols,
        dropped_corr_cols=dropped_corr_cols,
        final_feature_names=final_feature_names,
        static_cols=static_cols,
        dynamic_cols=dynamic_cols
    )
    return df_ok, y, groups, Xdf, artifacts


def preprocess_prediction(df_pred: pd.DataFrame, artifacts: PrepArtifacts) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    dfp = df_pred.copy()
    dfp.columns = sanitize_cols(dfp.columns)
    dfp = dfp.dropna(axis=0, how="all").reset_index(drop=True)

    missing = [c for c in artifacts.feature_cols_base if c not in dfp.columns]
    if missing:
        raise ValueError(f"预测文件缺少训练所需特征列(按列名匹配): {missing[:20]} ... (共{len(missing)}列)")

    Xdf_base = dfp.loc[:, artifacts.feature_cols_base].copy()
    Xdf_base = to_numeric_df(Xdf_base)

    if artifacts.allnan_cols_train:
        for c in artifacts.allnan_cols_train:
            if c in Xdf_base.columns:
                Xdf_base = Xdf_base.drop(columns=[c])

    for c in Xdf_base.columns:
        if Xdf_base[c].isna().any():
            Xdf_base[c] = Xdf_base[c].fillna(artifacts.medians_train_base.get(c, 0.0))

    Xdf = add_light_feature_engineering(Xdf_base)

    if artifacts.dropped_const_nzv_cols:
        drop_cols = [c for c in artifacts.dropped_const_nzv_cols if c in Xdf.columns]
        if drop_cols:
            Xdf = Xdf.drop(columns=drop_cols)

    if artifacts.dropped_corr_cols:
        drop_cols = [c for c in artifacts.dropped_corr_cols if c in Xdf.columns]
        if drop_cols:
            Xdf = Xdf.drop(columns=drop_cols)

    missing2 = [c for c in artifacts.final_feature_names if c not in Xdf.columns]
    if missing2:
        raise ValueError(f"预测集处理后仍缺少训练最终特征列: {missing2[:20]} ... (共{len(missing2)}列)")

    Xdf = Xdf[artifacts.final_feature_names].copy()

    X_static = Xdf[artifacts.static_cols].values.astype(float)
    X_dyn = Xdf[artifacts.dynamic_cols].values.astype(float)

    if artifacts.id_col in dfp.columns:
        city_arr = dfp[artifacts.id_col].astype(str).values
    else:
        city_arr = np.array([""] * len(dfp), dtype=object)

    return X_static, X_dyn, city_arr


# =========================
# 8) OOF by city
# =========================
def oof_eval_by_city(
    X_static: np.ndarray,
    X_dyn: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    params1: Dict,
    params2: Dict,
    fixed_rounds: int,
    w_mode: str,
    w_alpha: float,
    w_k: float,
    w_cap: float
) -> Tuple[Dict[str, float], pd.DataFrame]:
    gkf = GroupKFold(n_splits=OOF_FOLDS)
    y_log = np.log1p(y)

    oof_pred1_log = np.full_like(y_log, np.nan, dtype=float)
    oof_pred_final_log = np.full_like(y_log, np.nan, dtype=float)

    for tr, va in gkf.split(X_static, y, groups):
        Xs_tr, Xs_va = X_static[tr], X_static[va]
        Xd_tr, Xd_va = X_dyn[tr], X_dyn[va]
        y_tr, y_va = y[tr], y[va]
        ytr_log, yva_log = y_log[tr], y_log[va]

        wtr1 = make_weights(y_tr, w_mode, w_alpha, w_k, w_cap)
        bst1, it1 = train_xgb_log(
            params1, Xs_tr, ytr_log, Xs_va, yva_log, wtr1,
            num_round=min(MAX_BOOST_ROUNDS_CAP, fixed_rounds),
            early_stopping=EARLY_STOPPING
        )
        p1_va_log = bst1.predict(build_dmatrix(Xs_va), iteration_range=(0, it1))
        p1_tr_log = bst1.predict(build_dmatrix(Xs_tr), iteration_range=(0, it1))
        oof_pred1_log[va] = p1_va_log

        resid_tr = ytr_log - p1_tr_log
        resid_va = yva_log - p1_va_log
        X2_tr = np.c_[Xd_tr, p1_tr_log.reshape(-1, 1)]
        X2_va = np.c_[Xd_va, p1_va_log.reshape(-1, 1)]

        wtr2 = make_weights(y_tr, w_mode, w_alpha, w_k, w_cap)
        bst2, it2 = train_xgb_log(
            params2, X2_tr, resid_tr, X2_va, resid_va, wtr2,
            num_round=min(MAX_BOOST_ROUNDS_CAP, fixed_rounds),
            early_stopping=EARLY_STOPPING
        )
        p2_va_resid = bst2.predict(build_dmatrix(X2_va), iteration_range=(0, it2))
        oof_pred_final_log[va] = p1_va_log + p2_va_resid

    oof_pred1_raw = np.expm1(oof_pred1_log)
    oof_pred_final_raw = np.expm1(oof_pred_final_log)

    if CLIP_NONNEG:
        oof_pred1_raw = np.clip(oof_pred1_raw, 0.0, None)
        oof_pred_final_raw = np.clip(oof_pred_final_raw, 0.0, None)

    met1 = dict(
        raw_R2=float(r2_score(y, oof_pred1_raw)),
        log1p_R2=float(r2_score(y_log, oof_pred1_log)),
        MAE=float(mean_absolute_error(y, oof_pred1_raw)),
        RMSE=float(safe_rmse(y, oof_pred1_raw)),
    )
    metF = dict(
        raw_R2=float(r2_score(y, oof_pred_final_raw)),
        log1p_R2=float(r2_score(y_log, oof_pred_final_log)),
        MAE=float(mean_absolute_error(y, oof_pred_final_raw)),
        RMSE=float(safe_rmse(y, oof_pred_final_raw)),
    )

    oof_df = pd.DataFrame({
        "city": groups.astype(str),
        "y_true": y,
        "y_true_log1p": y_log,
        "stage1_oof_pred_log1p": oof_pred1_log,
        "stage1_oof_pred_raw": oof_pred1_raw,
        "final_oof_pred_log1p": oof_pred_final_log,
        "final_oof_pred_raw": oof_pred_final_raw,
    })
    return {"stage1": met1, "final": metF}, oof_df


# =========================
# 9) 预测
# =========================
def predict_two_stage(
    pred_csv_path: str,
    out_csv_path: str,
    artifacts: PrepArtifacts,
    bst1: xgb.Booster, it1: int,
    bst2: xgb.Booster, it2: int,
    beta: Optional[np.ndarray]
) -> pd.DataFrame:
    dfp = try_read_csv(pred_csv_path)
    Xs_p, Xd_p, city_arr = preprocess_prediction(dfp, artifacts)

    p1_log = bst1.predict(build_dmatrix(Xs_p), iteration_range=(0, it1))
    X2_p = np.c_[Xd_p, p1_log.reshape(-1, 1)]
    p2_resid = bst2.predict(build_dmatrix(X2_p), iteration_range=(0, it2))

    p_log = p1_log + p2_resid
    p_raw = np.expm1(p_log)
    if CLIP_NONNEG:
        p_raw = np.clip(p_raw, 0.0, None)

    if USE_CALIBRATION and (beta is not None):
        p_log, p_raw = apply_calibration(p_log, p_raw, beta, space=CALIBRATION_SPACE)
        if CLIP_NONNEG:
            p_raw = np.clip(p_raw, 0.0, None)

    out = pd.DataFrame({
        "city": city_arr,
        "y_pred_log1p": p_log,
        "y_pred_raw": p_raw,
    })
    p = safe_to_csv(out, out_csv_path, index=False, encoding="utf-8-sig")
    print(f"[Saved] {p}")
    return out


# =========================
# 10) Monte Carlo区间
# =========================
def build_residual_bins_from_oof(oof_df: pd.DataFrame, n_bins: int = 10) -> Tuple[List[dict], np.ndarray]:
    y_pred_log = oof_df["final_oof_pred_log1p"].to_numpy(dtype=float)
    resid_log = oof_df["y_true_log1p"].to_numpy(dtype=float) - y_pred_log

    qbin = pd.qcut(y_pred_log, q=n_bins, duplicates="drop")
    cats = qbin.cat.categories if hasattr(qbin, "cat") else qbin.categories

    edges = []
    for inter in cats:
        edges.append(float(inter.left))
    edges.append(float(cats[-1].right))
    bin_edges = np.array(edges, dtype=float)

    bins = []
    bin_idx = np.digitize(y_pred_log, bin_edges[1:-1], right=True)
    k = int(bin_idx.max() + 1)

    for i in range(k):
        m = (bin_idx == i)
        r = resid_log[m]
        if r.size < 10:
            r = resid_log.copy()
        bins.append({"i": i, "resid": r})
    return bins, bin_edges


def mc_intervals_from_bins(pred_df: pd.DataFrame, bins: List[dict], bin_edges: np.ndarray,
                           n_mc: int = 2000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    p_log = pred_df["y_pred_log1p"].to_numpy(dtype=float)

    bin_idx = np.digitize(p_log, bin_edges[1:-1], right=True)
    bin_idx = np.clip(bin_idx, 0, len(bins) - 1)

    n = len(p_log)
    p_mean = np.zeros(n, dtype=float)
    p_p025 = np.zeros(n, dtype=float)
    p_p25 = np.zeros(n, dtype=float)
    p_p50 = np.zeros(n, dtype=float)
    p_p75 = np.zeros(n, dtype=float)
    p_p975 = np.zeros(n, dtype=float)

    for j in range(n):
        r = bins[int(bin_idx[j])]["resid"]
        samp = rng.choice(r, size=n_mc, replace=True)

        sim_log = p_log[j] + samp
        sim_raw = np.expm1(sim_log)
        if CLIP_NONNEG:
            sim_raw = np.clip(sim_raw, 0.0, None)

        p_mean[j] = float(np.mean(sim_raw))
        p_p025[j] = float(np.quantile(sim_raw, 0.025))
        p_p25[j] = float(np.quantile(sim_raw, 0.25))
        p_p50[j] = float(np.quantile(sim_raw, 0.50))
        p_p75[j] = float(np.quantile(sim_raw, 0.75))
        p_p975[j] = float(np.quantile(sim_raw, 0.975))

    out = pred_df.copy()
    out["y_pred_raw_mc_mean"] = p_mean
    out["y_pred_raw_mc_p025"] = p_p025
    out["y_pred_raw_mc_p25"] = p_p25
    out["y_pred_raw_mc_p50"] = p_p50
    out["y_pred_raw_mc_p75"] = p_p75
    out["y_pred_raw_mc_p975"] = p_p975
    out["y_pred_raw_mc_ci95_width"] = out["y_pred_raw_mc_p975"] - out["y_pred_raw_mc_p025"]
    out["y_pred_raw_mc_iqr"] = out["y_pred_raw_mc_p75"] - out["y_pred_raw_mc_p25"]
    return out


def save_mc(pred_point_df: pd.DataFrame, oof_df: pd.DataFrame, out_path: str,
            n_bins: int = 10, n_mc: int = 2000, seed: int = 42):
    bins, edges = build_residual_bins_from_oof(oof_df, n_bins=n_bins)
    out = mc_intervals_from_bins(pred_point_df, bins, edges, n_mc=n_mc, seed=seed)
    p = safe_to_csv(out, out_path, index=False, encoding="utf-8-sig")
    print(f"[Saved] {p}")


# =========================
# 11) 主流程
# =========================
def main():
    ensure_dir(OUT_DIR)

    # 读取训练集
    df = try_read_csv(DATA_PATH)
    df_ok, y, groups, Xdf, artifacts = preprocess_training(df)

    feature_names = artifacts.final_feature_names
    static_cols, dynamic_cols = artifacts.static_cols, artifacts.dynamic_cols

    if (len(static_cols) < 2) or (len(dynamic_cols) < 2):
        print("[Warn] static/dynamic分组过少, 建议检查关键词或改为手动指定")
        print("static_cols:", static_cols)
        print("dynamic_cols:", dynamic_cols)

    X_static = Xdf[static_cols].values.astype(float)
    X_dyn = Xdf[dynamic_cols].values.astype(float)

    # new-city holdout
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE_CITIES, random_state=RANDOM_STATE)
    tr_idx, te_idx = next(gss.split(X_static, y, groups=groups))

    Xs_tr, Xs_te = X_static[tr_idx], X_static[te_idx]
    Xd_tr, Xd_te = X_dyn[tr_idx], X_dyn[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    g_tr, g_te = groups[tr_idx], groups[te_idx]

    print("===== DATA DIAG =====")
    print(f"n_samples = {len(y)} | n_features_total = {len(feature_names)}")
    print(f"static_features = {len(static_cols)} | dynamic_features = {len(dynamic_cols)}")
    print(f"new_city_holdout: train_rows = {len(tr_idx)} | test_rows = {len(te_idx)} | test_cities = {len(np.unique(g_te))}")
    print()

    # Optuna objective
    def objective(trial: optuna.Trial) -> float:
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=trial.number + RANDOM_STATE)
        itr, iva = next(gss2.split(Xs_tr, y_tr, groups=g_tr))

        Xs_itr, Xs_iva = Xs_tr[itr], Xs_tr[iva]
        Xd_itr, Xd_iva = Xd_tr[itr], Xd_tr[iva]
        y_itr, y_iva = y_tr[itr], y_tr[iva]

        params1 = make_xgb_params(trial, "s1")
        params2 = make_xgb_params(trial, "s2")
        fixed_rounds = trial.suggest_int("fixed_rounds", 200, 1800)

        w_mode = trial.suggest_categorical("WEIGHT_MODE", [WEIGHT_MODE])
        w_alpha = trial.suggest_float("w_alpha", W_ALPHA_RANGE[0], W_ALPHA_RANGE[1]) if w_mode == "pow" else 0.0
        w_k = trial.suggest_float("w_k", W_K_RANGE[0], W_K_RANGE[1]) if w_mode == "quantile" else 0.0
        w_cap = trial.suggest_float("WEIGHT_CAP", WEIGHT_CAP_RANGE[0], WEIGHT_CAP_RANGE[1])

        cv = eval_one_split_two_stage(
            params1=params1, params2=params2,
            X_static_tr=Xs_itr, X_static_va=Xs_iva,
            X_dyn_tr=Xd_itr, X_dyn_va=Xd_iva,
            y_tr_raw=y_itr, y_va_raw=y_iva,
            w_mode=w_mode, w_alpha=w_alpha, w_k=w_k, w_cap=w_cap,
            fixed_rounds=fixed_rounds,
            early_stopping=EARLY_STOPPING
        )

        trial.set_user_attr("raw_cv", cv.raw_r2)
        trial.set_user_attr("log_cv", cv.log_r2)
        trial.set_user_attr("med_iter", cv.best_iter)
        return cv.score

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True, group=True)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True)

    bt = study.best_trial
    best_score = float(bt.value)
    raw_cv = float(bt.user_attrs.get("raw_cv", np.nan))
    log_cv = float(bt.user_attrs.get("log_cv", np.nan))
    med_iter = int(bt.user_attrs.get("med_iter", -1))

    fixed_rounds = int(bt.params.get("fixed_rounds", 600))
    w_mode = bt.params.get("WEIGHT_MODE", WEIGHT_MODE)
    w_alpha = float(bt.params.get("w_alpha", 0.0)) if w_mode == "pow" else 0.0
    w_k = float(bt.params.get("w_k", 0.0)) if w_mode == "quantile" else 0.0
    w_cap = float(bt.params.get("WEIGHT_CAP", 10.0))

    params1_raw = {k.replace("s1_", ""): v for k, v in bt.params.items() if k.startswith("s1_")}
    params2_raw = {k.replace("s2_", ""): v for k, v in bt.params.items() if k.startswith("s2_")}
    final_params1 = finalize_params(params1_raw)
    final_params2 = finalize_params(params2_raw)

    print("===== BEST (Optuna, TWO-STAGE, group by city) =====")
    print(f"score(mix) = {best_score:.4f} | raw_cv = {raw_cv:.4f} | log_cv = {log_cv:.4f} | med_iter = {med_iter}")
    print(f"fixed_rounds = {fixed_rounds} | WEIGHT_MODE = {w_mode} | w_alpha = {w_alpha:.3f} | w_k = {w_k:.3f} | WEIGHT_CAP = {w_cap:.1f}")
    print(f"CALIBRATION = {USE_CALIBRATION} ({CALIBRATION_SPACE})")
    print()

    # OOF
    oof_metrics, oof_df = oof_eval_by_city(
        X_static=Xs_tr, X_dyn=Xd_tr,
        y=y_tr, groups=g_tr,
        params1=final_params1, params2=final_params2,
        fixed_rounds=fixed_rounds,
        w_mode=w_mode, w_alpha=w_alpha, w_k=w_k, w_cap=w_cap
    )

    m1 = oof_metrics["stage1"]
    mF = oof_metrics["final"]

    print("===== STAGE-WISE METRICS (TRAIN, OOF by city) =====")
    print(f"Stage1 only (OOF): raw_R2 = {m1['raw_R2']:.4f} | log1p_R2 = {m1['log1p_R2']:.4f} | MAE = {m1['MAE']:.3f} | RMSE = {m1['RMSE']:.3f}")
    print(f"Final two-stage (OOF): raw_R2 = {mF['raw_R2']:.4f} | log1p_R2 = {mF['log1p_R2']:.4f} | MAE = {mF['MAE']:.3f} | RMSE = {mF['RMSE']:.3f}")
    print()

    safe_to_csv(oof_df, os.path.join(OUT_DIR, "train_oof_predictions_by_city.csv"), index=False, encoding="utf-8-sig")

    # final training on training cities; eval on holdout
    ytr_log = np.log1p(y_tr)
    yte_log = np.log1p(y_te)

    wtr1 = make_weights(y_tr, w_mode, w_alpha, w_k, w_cap)
    bst1, it1 = train_xgb_log(
        final_params1, Xs_tr, ytr_log, Xs_te, yte_log, wtr1,
        num_round=min(MAX_BOOST_ROUNDS_CAP, fixed_rounds),
        early_stopping=EARLY_STOPPING
    )

    pred1_tr_log = bst1.predict(build_dmatrix(Xs_tr), iteration_range=(0, it1))
    pred1_te_log = bst1.predict(build_dmatrix(Xs_te), iteration_range=(0, it1))

    resid_tr = ytr_log - pred1_tr_log
    resid_te = yte_log - pred1_te_log

    X2_tr = np.c_[Xd_tr, pred1_tr_log.reshape(-1, 1)]
    X2_te = np.c_[Xd_te, pred1_te_log.reshape(-1, 1)]

    wtr2 = make_weights(y_tr, w_mode, w_alpha, w_k, w_cap)
    bst2, it2 = train_xgb_log(
        final_params2, X2_tr, resid_tr, X2_te, resid_te, wtr2,
        num_round=min(MAX_BOOST_ROUNDS_CAP, fixed_rounds),
        early_stopping=EARLY_STOPPING
    )

    pred2_te_resid = bst2.predict(build_dmatrix(X2_te), iteration_range=(0, it2))
    pred_te_log = pred1_te_log + pred2_te_resid
    pred_te_raw = np.expm1(pred_te_log)

    if CLIP_NONNEG:
        pred_te_raw = np.clip(pred_te_raw, 0.0, None)

    beta = None
    if USE_CALIBRATION:
        pred2_tr_resid = bst2.predict(build_dmatrix(X2_tr), iteration_range=(0, it2))
        pred_tr_log = pred1_tr_log + pred2_tr_resid
        pred_tr_raw = np.expm1(pred_tr_log)
        if CLIP_NONNEG:
            pred_tr_raw = np.clip(pred_tr_raw, 0.0, None)

        beta = calibrate_predictions(y_tr, pred_tr_log, pred_tr_raw, space=CALIBRATION_SPACE)
        pred_te_log, pred_te_raw = apply_calibration(pred_te_log, pred_te_raw, beta, space=CALIBRATION_SPACE)
        if CLIP_NONNEG:
            pred_te_raw = np.clip(pred_te_raw, 0.0, None)

        print(f"[Calibration] space = {CALIBRATION_SPACE} | beta = {beta}")

    pred1_te_raw = np.expm1(pred1_te_log)
    if CLIP_NONNEG:
        pred1_te_raw = np.clip(pred1_te_raw, 0.0, None)

    met_stage1 = dict(
        raw_R2=float(r2_score(y_te, pred1_te_raw)),
        log1p_R2=float(r2_score(yte_log, pred1_te_log)),
        MAE=float(mean_absolute_error(y_te, pred1_te_raw)),
        RMSE=float(safe_rmse(y_te, pred1_te_raw)),
    )
    met_final = dict(
        raw_R2=float(r2_score(y_te, pred_te_raw)),
        log1p_R2=float(r2_score(yte_log, pred_te_log)),
        MAE=float(mean_absolute_error(y_te, pred_te_raw)),
        RMSE=float(safe_rmse(y_te, pred_te_raw)),
    )

    print("===== STAGE-WISE METRICS (HOLDOUT NEW CITY) =====")
    print(f"Stage1 only: raw_R2 = {met_stage1['raw_R2']:.4f} | log1p_R2 = {met_stage1['log1p_R2']:.4f} | MAE = {met_stage1['MAE']:.3f} | RMSE = {met_stage1['RMSE']:.3f}")
    print(f"Final two-stage: raw_R2 = {met_final['raw_R2']:.4f} | log1p_R2 = {met_final['log1p_R2']:.4f} | MAE = {met_final['MAE']:.3f} | RMSE = {met_final['RMSE']:.3f}")
    print()

    holdout_df = pd.DataFrame({
        "idx": te_idx,
        "city": df_ok.iloc[te_idx][artifacts.id_col].astype(str).values,
        "y_true": y_te,
        "y_true_log1p": np.log1p(y_te),
        "stage1_pred_log1p": pred1_te_log,
        "stage1_pred_raw": pred1_te_raw,
        "final_pred_log1p": pred_te_log,
        "final_pred_raw": pred_te_raw,
        "abs_err_final": np.abs(y_te - pred_te_raw),
    })
    safe_to_csv(holdout_df, os.path.join(OUT_DIR, "holdout_predictions_newcity.csv"), index=False, encoding="utf-8-sig")

    # all predictions on all rows
    pred1_all_log = bst1.predict(build_dmatrix(X_static), iteration_range=(0, it1))
    X2_all = np.c_[X_dyn, pred1_all_log.reshape(-1, 1)]
    pred2_all_resid = bst2.predict(build_dmatrix(X2_all), iteration_range=(0, it2))
    pred_all_log = pred1_all_log + pred2_all_resid
    pred_all_raw = np.expm1(pred_all_log)

    if CLIP_NONNEG:
        pred_all_raw = np.clip(pred_all_raw, 0.0, None)

    if USE_CALIBRATION and (beta is not None):
        pred_all_log, pred_all_raw = apply_calibration(pred_all_log, pred_all_raw, beta, space=CALIBRATION_SPACE)
        if CLIP_NONNEG:
            pred_all_raw = np.clip(pred_all_raw, 0.0, None)

    all_df = pd.DataFrame({
        "idx": np.arange(len(y)),
        "city": df_ok[artifacts.id_col].astype(str).values,
        "y_true": y,
        "y_true_log1p": np.log1p(y),
        "final_pred_log1p": pred_all_log,
        "final_pred_raw": pred_all_raw,
    })
    safe_to_csv(all_df, os.path.join(OUT_DIR, "all_predictions.csv"), index=False, encoding="utf-8-sig")

    metrics_summary = pd.DataFrame([
        {"split": "train_oof_by_city", "model": "stage1", "raw_R2": m1["raw_R2"], "log1p_R2": m1["log1p_R2"], "MAE": m1["MAE"], "RMSE": m1["RMSE"]},
        {"split": "train_oof_by_city", "model": "final_two_stage", "raw_R2": mF["raw_R2"], "log1p_R2": mF["log1p_R2"], "MAE": mF["MAE"], "RMSE": mF["RMSE"]},
        {"split": "holdout_new_city", "model": "stage1", "raw_R2": met_stage1["raw_R2"], "log1p_R2": met_stage1["log1p_R2"], "MAE": met_stage1["MAE"], "RMSE": met_stage1["RMSE"]},
        {"split": "holdout_new_city", "model": "final_two_stage", "raw_R2": met_final["raw_R2"], "log1p_R2": met_final["log1p_R2"], "MAE": met_final["MAE"], "RMSE": met_final["RMSE"]},
    ])
    safe_to_csv(metrics_summary, os.path.join(OUT_DIR, "metrics_summary_stagewise.csv"), index=False, encoding="utf-8-sig")

    # 保存模型和工件
    bst1.save_model(os.path.join(OUT_DIR, "stage1_model.json"))
    bst2.save_model(os.path.join(OUT_DIR, "stage2_model.json"))

    artifacts_json = {
        "prep_artifacts": asdict(artifacts),
        "CLIP_NONNEG": CLIP_NONNEG,
        "USE_CALIBRATION": USE_CALIBRATION,
        "CALIBRATION_SPACE": CALIBRATION_SPACE,
        "beta": beta.tolist() if (beta is not None) else None,
        "it1": int(it1),
        "it2": int(it2),
        "best_trial_value": best_score,
        "best_trial_params": bt.params,
    }
    with open(os.path.join(OUT_DIR, "train_artifacts.json"), "w", encoding="utf-8") as f:
        json.dump(artifacts_json, f, ensure_ascii=False, indent=2)

    print("Saved: stage1_model.json, stage2_model.json, train_artifacts.json")
    print()

    # 训练后直接预测 2015 / 2017 / 2019 / 2021 / 2023
    if RUN_PREDICTION_AFTER_TRAINING:
        pred_list = [
            (PRED_2015, "predictions_2015_monthly.csv"),
            (PRED_2017, "predictions_2017_monthly.csv"),
            (PRED_2019, "predictions_2019_monthly.csv"),
            (PRED_2021, "predictions_2021_monthly.csv"),
            (PRED_2023, "predictions_2023_monthly.csv"),
        ]

        for pred_path, out_name in pred_list:
            out_path = os.path.join(OUT_DIR, out_name)

            pred_point_df = predict_two_stage(
                pred_csv_path=pred_path,
                out_csv_path=out_path,
                artifacts=artifacts,
                bst1=bst1, it1=it1,
                bst2=bst2, it2=it2,
                beta=beta
            )

            if RUN_MC_AFTER_PRED:
                out_ci = os.path.join(OUT_DIR, out_name.replace(".csv", "_with_ci_mc.csv"))
                save_mc(
                    pred_point_df=pred_point_df,
                    oof_df=oof_df,
                    out_path=out_ci,
                    n_bins=MC_N_BINS,
                    n_mc=MC_N,
                    seed=MC_SEED
                )

    print("Done.")


if __name__ == "__main__":
    main()


[I 2026-11-11 16:51:55,716] A new study created in memory with name: no-name-5e8dd899-2b92-4520-9aa3-f8db92e29023


===== DATA DIAG =====
n_samples = 205 | n_features_total = 32
static_features = 17 | dynamic_features = 15
new_city_holdout: train_rows = 164 | test_rows = 41 | test_cities = 41



  0%|          | 0/300 [00:00<?, ?it/s]

[I 2026-11-11 16:51:55,909] Trial 0 finished with value: 0.46453820842697024 and parameters: {'s1_grow_policy': 'lossguide', 's1_max_depth': 4, 's1_max_leaves': 473, 's1_use_alpha': 0, 's1_eta': 0.004822335219684796, 's1_subsample': 0.9031616510212273, 's1_colsample_bytree': 0.7904460046972835, 's1_colsample_bylevel': 0.8332290311184182, 's1_colsample_bynode': 0.558233797718321, 's1_min_child_weight': 26.90038038390857, 's1_gamma': 6.659541126403374, 's1_max_delta_step': 2.548069328139314, 's1_lambda_': 0.639981257861713, 's1_max_bin': 198, 's2_grow_policy': 'lossguide', 's2_max_depth': 2, 's2_max_leaves': 246, 's2_use_alpha': 0, 's2_eta': 0.010243790402910097, 's2_subsample': 0.7282266451527921, 's2_colsample_bytree': 0.7324279936868143, 's2_colsample_bylevel': 0.8640703845572054, 's2_colsample_bynode': 0.6298695128633439, 's2_min_child_weight': 5.158353157216827, 's2_gamma': 4.73931655089634, 's2_max_delta_step': 0.5574049526399727, 's2_lambda_': 9.747236096769878, 's2_max_bin': 193,